# 19. 连续控制与现代深度强化学习体系
在连续动作空间中，值函数方法面临一个直接困难：动作不再是有限集合，$\max_a Q(s,a)$ 不能像离散动作那样直接枚举。于是现代连续控制算法大多转向 Actor-Critic 框架，让 Actor 直接输出连续动作，让 Critic 负责评价。

DDPG、TD3 与 SAC 是这条路线最核心的三种代表。它们既共享离策略、经验回放与目标网络的工程基座，又在过估计控制、探索机制与熵正则化上形成了清晰分工。

## 连续控制中的离策略 Actor-Critic的形式化定义

            连续控制问题通常仍定义在 MDP 上，但动作空间变为连续域 $a \in \mathbb{R}^d$。此时确定性策略梯度（Deterministic Policy Gradient, DPG）给出
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{s\sim\mathcal{D}}\left[
    \nabla_a Q_\phi(s,a)\vert_{a=\mu_\theta(s)} \nabla_\theta \mu_\theta(s)
\right].
$$
DDPG 用神经网络实现这一思想；TD3 通过双 Critic 与目标平滑修正 DDPG 的偏差问题；SAC 则把最大熵目标引入 Actor-Critic，使策略在优化回报的同时保持充分随机性。

## 输入、输出与参数化方式

            输入是连续状态特征，Actor 输出连续动作或动作分布参数，Critic 输出标量价值估计：

- DDPG / TD3：Actor 常输出确定性动作 $\mu_\theta(s)$；
- SAC：Actor 输出随机策略 $\pi_\theta(a\mid s)$，通常为高斯分布再经 squashing。

在工程实现中，Replay Buffer、目标网络、Polyak averaging 与动作范围裁剪是必不可少的基础组件。

## 结构分解与信息流

            DDPG 的信息流为：用 Actor 产生动作，用 Critic 评估动作，再沿 Critic 对动作的梯度方向更新 Actor。TD3 进一步增加两个关键修正：

1. **Clipped Double Q**：两个 Critic 取最小值，减轻过估计；
2. **Target Policy Smoothing**：给目标动作加小噪声，降低 Critic 对尖锐误差的过拟合；
3. **Delayed Policy Update**：Critic 多更新几步后再更新 Actor。

SAC 则把目标改成最大熵 RL：
$$
J(\pi)=\sum_t \mathbb{E}[r(s_t,a_t)+\alpha \mathcal{H}(\pi(\cdot\mid s_t))].
$$
因而策略不仅追求高回报，也追求合适的随机性和覆盖度。

## 优化目标与训练机制

            三类方法的目标可以概括为：

- **DDPG / TD3 Critic**：
  $$
  \mathcal{L}_Q = \mathbb{E}\left[\left(Q_\phi(s,a) - y\right)^2\right]
  $$
  $$
  y = r + \gamma Q_{\bar{\phi}}(s', \mu_{\bar{\theta}}(s'))
  $$

- **DDPG / TD3 Actor**：
  $$
  \mathcal{L}_{\text{actor}} = -\mathbb{E}_{s\sim \mathcal{D}}[Q_\phi(s,\mu_\theta(s))]
  $$

- **SAC Actor**：
  $$
  \mathcal{L}_{\pi} = \mathbb{E}_{s\sim\mathcal{D}, a\sim\pi_\theta}
  \left[\alpha \log \pi_\theta(a\mid s) - \min_j Q_{\phi_j}(s,a)\right]
  $$

熵系数 $\alpha$ 控制探索与利用的权衡，是 SAC 的关键超参数。

## 计算复杂度、统计性质与工程代价

            连续控制算法的核心难点不是网络规模，而是 Critic 误差被 Actor 放大后形成的反馈环。只要 Critic 有系统性偏差，Actor 就会主动把策略推向被误估的动作区域。这正是 TD3 和 SAC 相比 DDPG 更稳定的根本原因。

从样本效率看，这类算法通常优于 on-policy 的 PPO；但对奖励尺度、目标网络更新率、探索噪声和归一化策略非常敏感。

## 与相邻模型的差异

DDPG 适合作为连续控制的最小离策略基线；TD3 适合在确定性策略框架下追求更高稳定性；SAC 在实践中通常最稳健，尤其适合奖励噪声大、探索困难或多峰动作分布的场景，但其实现和调参复杂度也更高。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5.5)

blocks = [
    (1.4, 2.75, 1.6, 1.0, "#4C78A8", "state s"),
    (4.2, 3.7, 2.2, 1.0, "#F58518", "actor\npi(a|s)"),
    (4.2, 1.8, 2.2, 1.0, "#54A24B", "critic\nV(s) / Q(s,a)"),
    (7.2, 2.75, 2.0, 1.0, "#E45756", "action a"),
    (10.2, 2.75, 2.3, 1.2, "#72B7B2", "advantage / TD error"),
]
for x, y, w, h, color, text in blocks:
    rect = plt.Rectangle((x - w / 2, y - h / 2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(3.1, 3.7), xytext=(2.2, 2.95), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(3.1, 1.8), xytext=(2.2, 2.55), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(6.2, 2.75), xytext=(5.3, 3.35), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(9.0, 2.75), xytext=(8.2, 2.75), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(5.3, 2.15), xytext=(9.1, 2.25), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.text(8.2, 4.45, "critic supplies the learning signal for the actor", ha="center", fontsize=11)
ax.set_title("Actor-Critic Information Flow")
plt.show()

## 何时使用 DDPG、TD3、SAC

- 若任务结构简单、需要快速构建最小可运行基线，可先从 DDPG 出发。
- 若 DDPG 出现明显过估计、策略抖动或训练发散，TD3 往往是第一选择。
- 若需要更稳健的探索、更强的多样性保持或更平滑的训练过程，SAC 通常是更可靠的主力方案。

在机器人控制、连续参数调节和高维动作优化场景中，SAC 已成为最常见的通用基线之一。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
from collections import deque
import copy

def optimal_action_np(states):
    return np.tanh(np.sin(np.pi * states))

def reward_fn_np(states, actions):
    return 1.0 - (actions - optimal_action_np(states)) ** 2

def optimal_action_torch(states):
    return torch.tanh(torch.sin(torch.tensor(np.pi, dtype=states.dtype) * states))

def reward_fn_torch(states, actions):
    return 1.0 - (actions - optimal_action_torch(states)) ** 2

class DeterministicActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.net(x)

class GaussianActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )
        self.mean = nn.Linear(32, 1)
        self.log_std = nn.Linear(32, 1)

    def forward(self, x):
        h = self.backbone(x)
        mean = torch.tanh(self.mean(h))
        log_std = torch.clamp(self.log_std(h), -2.5, -0.2)
        return mean, log_std

    def sample(self, x):
        mean, log_std = self.forward(x)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        z = normal.rsample()
        action = torch.tanh(z)
        log_prob = normal.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
        return action, log_prob.sum(dim=1), mean

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, states, actions):
        x = torch.cat([states, actions], dim=1)
        return self.net(x).squeeze(-1)

def train_ddpg_style(steps=900, batch_size=128):
    actor = DeterministicActor()
    critic = Critic()
    actor_targ = copy.deepcopy(actor)
    critic_targ = copy.deepcopy(critic)
    actor_opt = torch.optim.Adam(actor.parameters(), lr=1e-3)
    critic_opt = torch.optim.Adam(critic.parameters(), lr=1e-3)
    replay = deque(maxlen=5000)
    tau = 0.02
    reward_curve = []

    for step in range(steps):
        state = np.random.uniform(-1.0, 1.0, size=(1, 1)).astype(np.float32)
        with torch.no_grad():
            action = actor(torch.tensor(state)).numpy()
        action = np.clip(action + np.random.normal(0, 0.15, size=action.shape), -1.0, 1.0)
        reward = reward_fn_np(state, action).astype(np.float32)
        next_state = np.random.uniform(-1.0, 1.0, size=(1, 1)).astype(np.float32)
        replay.append((state[0], action[0], reward[0], next_state[0], 1.0))

        if len(replay) >= batch_size:
            batch = random.sample(replay, batch_size)
            states_b = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
            actions_b = torch.tensor(np.array([b[1] for b in batch]), dtype=torch.float32)
            rewards_b = torch.tensor(np.array([b[2] for b in batch]).reshape(-1), dtype=torch.float32)

            q_values = critic(states_b, actions_b)
            critic_loss = nn.functional.mse_loss(q_values, rewards_b)
            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()

            actor_actions = actor(states_b)
            actor_loss = -critic(states_b, actor_actions).mean()
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()

            for online, target in zip(actor.parameters(), actor_targ.parameters()):
                target.data.mul_(1 - tau).add_(tau * online.data)
            for online, target in zip(critic.parameters(), critic_targ.parameters()):
                target.data.mul_(1 - tau).add_(tau * online.data)

        if (step + 1) % 20 == 0:
            eval_states = torch.linspace(-1, 1, 200).unsqueeze(1)
            with torch.no_grad():
                eval_actions = actor(eval_states)
                eval_reward = reward_fn_torch(eval_states, eval_actions).mean().item()
            reward_curve.append({"step": step + 1, "algorithm": "DDPG-style", "avg_reward": eval_reward})
    return actor, pd.DataFrame(reward_curve)

def train_sac_style(steps=900, batch_size=128, alpha=0.15):
    actor = GaussianActor()
    q1 = Critic()
    q2 = Critic()
    actor_opt = torch.optim.Adam(actor.parameters(), lr=1e-3)
    q1_opt = torch.optim.Adam(q1.parameters(), lr=1e-3)
    q2_opt = torch.optim.Adam(q2.parameters(), lr=1e-3)
    replay = deque(maxlen=5000)
    reward_curve = []

    for step in range(steps):
        state = np.random.uniform(-1.0, 1.0, size=(1, 1)).astype(np.float32)
        with torch.no_grad():
            action, _, _ = actor.sample(torch.tensor(state))
        action_np = action.numpy()
        reward = reward_fn_np(state, action_np).astype(np.float32)
        next_state = np.random.uniform(-1.0, 1.0, size=(1, 1)).astype(np.float32)
        replay.append((state[0], action_np[0], reward[0], next_state[0], 1.0))

        if len(replay) >= batch_size:
            batch = random.sample(replay, batch_size)
            states_b = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
            actions_b = torch.tensor(np.array([b[1] for b in batch]), dtype=torch.float32)
            rewards_b = torch.tensor(np.array([b[2] for b in batch]).reshape(-1), dtype=torch.float32)

            q1_loss = nn.functional.mse_loss(q1(states_b, actions_b), rewards_b)
            q2_loss = nn.functional.mse_loss(q2(states_b, actions_b), rewards_b)
            q1_opt.zero_grad()
            q1_loss.backward()
            q1_opt.step()
            q2_opt.zero_grad()
            q2_loss.backward()
            q2_opt.step()

            sampled_actions, log_prob, _ = actor.sample(states_b)
            q_min = torch.minimum(q1(states_b, sampled_actions), q2(states_b, sampled_actions))
            actor_loss = (alpha * log_prob - q_min).mean()
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()

        if (step + 1) % 20 == 0:
            eval_states = torch.linspace(-1, 1, 200).unsqueeze(1)
            with torch.no_grad():
                eval_actions, _, means = actor.sample(eval_states)
                eval_reward = reward_fn_torch(eval_states, eval_actions).mean().item()
            reward_curve.append({"step": step + 1, "algorithm": "SAC-style", "avg_reward": eval_reward})
    return actor, pd.DataFrame(reward_curve)

ddpg_actor, ddpg_curve = train_ddpg_style()
sac_actor, sac_curve = train_sac_style()
control_df = pd.concat([ddpg_curve, sac_curve], ignore_index=True)
control_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=control_df, x="step", y="avg_reward", hue="algorithm", marker="o")
plt.title("Continuous Control on a Synthetic Contextual Bandit")
plt.ylabel("average reward on evaluation grid")
plt.show()

In [ ]:
state_grid = torch.linspace(-1, 1, 200).unsqueeze(1)
with torch.no_grad():
    ddpg_actions = ddpg_actor(state_grid).numpy().reshape(-1)
    _, _, sac_means = sac_actor.sample(state_grid)
    sac_actions = sac_means.numpy().reshape(-1)

curve_df = pd.DataFrame(
    {
        "state": state_grid.numpy().reshape(-1),
        "optimal": optimal_action_np(state_grid.numpy().reshape(-1)),
        "DDPG-style": ddpg_actions,
        "SAC-style mean": sac_actions,
    }
)
curve_long = curve_df.melt(id_vars="state", var_name="policy", value_name="action")
curve_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=curve_long, x="state", y="action", hue="policy")
plt.title("Learned Continuous Policies vs Optimal Action Curve")
plt.ylabel("action")
plt.show()

true_q = np.array([1.2, 1.1, 0.95])
single_q, clipped_double_q = [], []
for _ in range(10000):
    q_a = true_q + np.random.normal(0, 0.35, size=3)
    q_b = true_q + np.random.normal(0, 0.35, size=3)
    single_q.append(np.max(q_a))
    clipped_double_q.append(np.max(np.minimum(q_a, q_b)))
bias_compare_df = pd.DataFrame(
    {
        "estimator": ["single critic max", "clipped double max", "true max"],
        "value": [np.mean(single_q), np.mean(clipped_double_q), np.max(true_q)],
    }
)

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.barplot(data=bias_compare_df, x="estimator", y="value", palette="crest")
plt.title("Why TD3 Uses Clipped Double Critics")
plt.show()

## 场景匹配建议

- **DDPG**：适合做确定性策略与连续控制的入门基线，但对超参数和 Critic 偏差敏感。
- **TD3**：当 DDPG 出现明显过估计、Q 值爆炸、策略被单点误差带偏时，TD3 往往是更稳的替代。
- **SAC**：当探索困难、动作分布多峰、奖励噪声较大或需要更稳定泛化时，优先考虑 SAC。

在工业控制或机器人任务中，SAC 和 TD3 常常是默认首选；PPO 则更多用于 on-policy、仿真稳定或并行采样容易的场景。

## 主要资料

- DDPG（2015）: https://arxiv.org/abs/1509.02971
- TD3（2018）: https://arxiv.org/abs/1802.09477
- SAC（2018）: https://arxiv.org/abs/1801.01290